# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Srinadh2314/srinadh-flyrank-intership/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

**Baseline rule: Stale + Visible**

I will prioritize pages that have not been updated recently and still receive meaningful search exposure. The rule uses two observable signals: `days_since_last_update` for staleness and `impressions_90d` for visibility.

**Signal verdicts**

- `days_since_last_update`: **MIXED** — the declining rate increases through the 91–180 day bucket but does not continue increasing for older buckets.
- `impressions_90d`: **CONFIRMED** — pages with 101+ impressions show a higher declining rate than pages with 1–100 impressions.

**Reason code:** `STALE_VISIBLE`

**Action label:** `REVIEW_REFRESH`

The score is decision-support for human review, not an automatic instruction to change the page.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np

csv_url = "https://raw.githubusercontent.com/Srinadh2314/srinadh-flyrank-intership/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(csv_url)

print("Rows:", len(df))

# Signal 1: staleness
df["stale_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=[-np.inf, 30, 90, 180, 365, np.inf],
    labels=["0-30", "31-90", "91-180", "181-365", "366+"]
)

stale_table = (
    df.groupby("stale_bucket", observed=False)
      .agg(
          n=("days_since_last_update", "size"),
          mean_declining=("trend_direction", lambda s: (s.str.lower() == "down").mean())
      )
      .reset_index()
)

print("\nSignal 1 — days_since_last_update")
display(stale_table)

# Signal 2: visibility
df["visibility_bucket"] = pd.cut(
    df["impressions_90d"],
    bins=[-np.inf, 0, 100, 500, 1000, np.inf],
    labels=["0", "1-100", "101-500", "501-1000", "1000+"]
)

visibility_table = (
    df.groupby("visibility_bucket", observed=False)
      .agg(
          n=("impressions_90d", "size"),
          mean_declining=("trend_direction", lambda s: (s.str.lower() == "down").mean())
      )
      .reset_index()
)

print("\nSignal 2 — impressions_90d")
display(visibility_table)

Rows: 30000

Signal 1 — days_since_last_update


,stale_bucket,n,mean_declining
0,0-30,20480,0.511377
1,31-90,175,0.588571
2,91-180,9171,0.611057
3,181-365,169,0.467456
4,366+,5,0.600000



Signal 2 — impressions_90d


,visibility_bucket,n,mean_declining
0,0,0,NaN
1,1-100,8006,0.389208
2,101-500,5279,0.604281
3,501-1000,3206,0.600437
4,1000+,13509,0.594493


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Build ONE baseline rule:
# stale + visible pages receive higher refresh-review priority.

df["stale"] = (df["days_since_last_update"] >= 180).astype(int)
df["visible"] = (df["impressions_90d"] >= 500).astype(int)

# Higher score = higher priority.
df["baseline_score"] = (
    df["stale"] *
    df["visible"] *
    df["impressions_90d"]
)

df["reason_code"] = np.where(
    df["baseline_score"] > 0,
    "STALE_VISIBLE",
    "NONE"
)

df["action"] = np.where(
    df["baseline_score"] > 0,
    "REVIEW_REFRESH",
    "NO_ACTION"
)

queue = (
    df.sort_values(
        ["baseline_score", "impressions_90d"],
        ascending=[False, False]
    )
    .reset_index(drop=True)
)

queue["rank"] = np.arange(1, len(queue) + 1)

output_cols = [
    "rank",
    "baseline_score",
    "reason_code",
    "action",
    "impressions_90d",
    "days_since_last_update",
    "avg_position",
    "ctr",
    "trend_direction"
]

baseline_queue = queue[output_cols].copy()

print("Ranked queue rows:", len(baseline_queue))
display(baseline_queue.head(20))

# Write the required CSV
import os
os.makedirs("work/outputs", exist_ok=True)

baseline_queue.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print("\nSaved:")
print("work/outputs/baseline_action_score.csv")


Ranked queue rows: 30000


,rank,baseline_score,reason_code,action,impressions_90d,days_since_last_update,avg_position,ctr,trend_direction
0,1,61678,STALE_VISIBLE,REVIEW_REFRESH,61678,194,19.7,0.15,down
1,2,59472,STALE_VISIBLE,REVIEW_REFRESH,59472,194,24.8,0.13,down
2,3,25715,STALE_VISIBLE,REVIEW_REFRESH,25715,194,22.2,0.23,down
3,4,13299,STALE_VISIBLE,REVIEW_REFRESH,13299,193,10.5,0.49,down
4,5,7812,STALE_VISIBLE,REVIEW_REFRESH,7812,194,39.0,0.01,down
5,6,7558,STALE_VISIBLE,REVIEW_REFRESH,7558,193,17.9,0.20,down
6,7,4590,STALE_VISIBLE,REVIEW_REFRESH,4590,194,31.0,0.00,down
7,8,4556,STALE_VISIBLE,REVIEW_REFRESH,4556,194,16.4,0.33,down
8,9,4429,STALE_VISIBLE,REVIEW_REFRESH,4429,194,25.3,0.38,down
9,10,1697,STALE_VISIBLE,REVIEW_REFRESH,1697,193,15.8,0.12,down



Saved:
work/outputs/baseline_action_score.csv


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*
**Top-20 review**

The first 17 ranked rows receive the `REVIEW_REFRESH` action because they are both stale and visible under the baseline rule. Their priority is driven mainly by impressions while satisfying the staleness threshold. Ranks 18–20 have a score of zero and `NO_ACTION`; they appear in the top 20 only because many rows are tied at zero, showing a weakness of the baseline ranking.

| Rank | Action | Reason code | Confidence note | What would make it wrong |
|---|---|---|---|---|
| 1 | REVIEW_REFRESH | STALE_VISIBLE | High — very high impressions and 194 days since update. | The page may be intentionally evergreen or already scheduled for review. |
| 2 | REVIEW_REFRESH | STALE_VISIBLE | High — very high impressions and 194 days since update. | High visibility may not mean the page needs a refresh. |
| 3 | REVIEW_REFRESH | STALE_VISIBLE | High — strong visibility and 194 days since update. | The page may still be performing acceptably despite being old. |
| 4 | REVIEW_REFRESH | STALE_VISIBLE | High — 13,299 impressions and 193 days since update. | The observed signals may reflect a temporary condition rather than a refresh opportunity. |
| 5 | REVIEW_REFRESH | STALE_VISIBLE | High — 7,812 impressions and 194 days since update. | Low average position may have a different cause than content staleness. |
| 6 | REVIEW_REFRESH | STALE_VISIBLE | High — 7,558 impressions and 193 days since update. | A refresh may not address the underlying reason for performance. |
| 7 | REVIEW_REFRESH | STALE_VISIBLE | High — 4,590 impressions and 194 days since update. | Zero CTR could reflect measurement or query-specific effects rather than content quality. |
| 8 | REVIEW_REFRESH | STALE_VISIBLE | High — 4,556 impressions and 194 days since update. | The page may already be adequate for its search intent. |
| 9 | REVIEW_REFRESH | STALE_VISIBLE | High — 4,429 impressions and 194 days since update. | A high CTR can indicate that the existing page is already effective. |
| 10 | REVIEW_REFRESH | STALE_VISIBLE | High — 1,697 impressions and 193 days since update. | The page may not have enough strategic value to justify review time. |
| 11 | REVIEW_REFRESH | STALE_VISIBLE | Medium — 1,408 impressions and 183 days since update. | Being stale and visible alone does not prove that a refresh is needed. |
| 12 | REVIEW_REFRESH | STALE_VISIBLE | Medium — 1,316 impressions and 194 days since update, but trend is stable. | Stable trend suggests the page may not currently need intervention. |
| 13 | REVIEW_REFRESH | STALE_VISIBLE | Medium — 954 impressions and 301 days since update. | Very old content can still be intentionally maintained and useful. |
| 14 | REVIEW_REFRESH | STALE_VISIBLE | Medium — 828 impressions and 194 days since update. | Moderate exposure may not justify spending review time. |
| 15 | REVIEW_REFRESH | STALE_VISIBLE | Medium — 821 impressions and 301 days since update. | Old age alone does not establish that updating will help. |
| 16 | REVIEW_REFRESH | STALE_VISIBLE | Medium — 545 impressions and 183 days since update. | The page may have limited opportunity despite meeting the rule. |
| 17 | REVIEW_REFRESH | STALE_VISIBLE | Medium — 533 impressions and 183 days since update, with average position 48.0. | Poor position may be caused by factors unrelated to content freshness. |
| 18 | NO_ACTION | NONE | Low — score is zero despite high impressions. | The tie at zero means this page is not actually prioritized by the rule and should not be interpreted as a top opportunity. |
| 19 | NO_ACTION | NONE | Low — score is zero and the page is not stale. | Its appearance in the top 20 is caused by score ties, not by evidence for refresh. |
| 20 | NO_ACTION | NONE | Low — score is zero and the page is not stale. | The page is only present because many rows share the same zero score. |

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*
**Weak picks**

The weakest picks are ranks 18–20. They have `baseline_score = 0` and `reason_code = NONE`, so they are not genuine refresh recommendations. Their appearance in the top 20 is caused by many tied zero scores and the secondary sort on impressions. This shows that the baseline rule is useful for identifying stale and visible pages but is weak at ordering pages outside that group.

**Leakage check**

The baseline uses only `days_since_last_update` and `impressions_90d`, which are observable pre-decision signals. It does not use `trend_direction`, `trend_pct`, `is_declining_label`, or future-window data. Therefore, no label-derived or future-window input is used.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Leakage check: confirm only pre-decision signals are used.

rule_features = ["days_since_last_update", "impressions_90d"]

forbidden_features = [
    "trend_direction",
    "trend_pct",
    "is_declining_label",
    "march_impressions"
]

print("Rule features:", rule_features)
print("Forbidden features present in rule:")
for feature in forbidden_features:
    print(f"{feature}: {feature in df.columns and feature in rule_features}")

print("\nWeak picks:")
display(
    baseline_queue[
        baseline_queue["baseline_score"] == 0
    ].head(10)
)


Rule features: ['days_since_last_update', 'impressions_90d']
Forbidden features present in rule:
trend_direction: False
trend_pct: False
is_declining_label: False
march_impressions: False

Weak picks:


,rank,baseline_score,reason_code,action,impressions_90d,days_since_last_update,avg_position,ctr,trend_direction
17,18,0,NONE,NO_ACTION,517715,104,4.2,0.14,down
18,19,0,NONE,NO_ACTION,517109,22,5.4,0.25,stable
19,20,0,NONE,NO_ACTION,509252,20,2.5,0.15,down
20,21,0,NONE,NO_ACTION,497727,48,22.2,0.10,up
21,22,0,NONE,NO_ACTION,463103,20,2.3,0.41,down
22,23,0,NONE,NO_ACTION,443434,104,27.9,0.21,stable
23,24,0,NONE,NO_ACTION,416180,22,4.0,0.23,down
24,25,0,NONE,NO_ACTION,347399,104,4.2,0.53,down
25,26,0,NONE,NO_ACTION,345111,20,5.4,0.21,up
26,27,0,NONE,NO_ACTION,312694,20,1.4,0.65,stable


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.